# Classification 종합실습

 ## 신용대출 심사

* 고객사는 ## 은행입니다. 신용평가 업무를 인공지능으로 전환하고자 여러분에게 모델링을 의뢰하였습니다.
* 대출업무는
    * 은행 창구에서 신청을 받고
    * 본사의 심사부서에서는 신용평가를 통해 대출 신청에 대한 승인 여부를 결정해 왔습니다.

* 현장의 요구
    * 경쟁사의 공격적인 대출상품 판매로, 본사에서는 자사 은행의 대출 실적이 줄어들고 있는 것에 부담을 느끼고 있습니다.
    * 그런데, 자사 은행에서는 신용평가 결과의 정확성에 의문을 품고 있으며, 신용평가 기준을 완화하여 가급적 대출승인 범위를 더 확대해 주기를 요구합니다. 

* 신용평가 업무를 인공지능으로 전환
    * 현장의 요구를 감안하여, 과거 사람이 하던 평가방식을 개선하고자 인공지능에 의한 예측 모델을 만들고, 정확도를 높이고자 합니다.
    * 최적의 모델을 생성해 봅시다.


## 1.환경준비

### (1) import

In [1]:
#라이브러리들을 불러오자.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# 모델링
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import * 
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

import warnings    # 경고메시지 제외
warnings.filterwarnings(action='ignore')

### (2) 데이터 준비

In [2]:
path = 'https://raw.githubusercontent.com/DA4BAM/dataset/master/credit_all.csv'
data = pd.read_csv(path)
data.loc[data['Payment'] == 4, 'Payment'] = 3
data.head()

,Creditability,AccountBalance,Duration,Payment,Purpose,CreditAmount,Employment,SexMarital,CurrentAddress,MostValuableAsset,Age,Apartment,CreditCount,Occupation,Dependents,Telephone,ForeignWorker
0,1,3,24,2,0,1249,2,4,2,1,28,2,1,3,1,1,1
1,1,2,9,2,0,276,3,4,4,1,22,1,1,2,1,1,1
2,1,1,18,3,2,1049,2,2,4,2,21,1,1,3,1,1,1
3,1,1,24,3,1,6419,5,2,4,4,44,3,2,4,2,2,1
4,1,3,12,2,2,1424,5,2,4,1,55,2,1,4,1,2,1


|	칼럼명	|	설명	|	 	|	값 의미	|
|	-----	|	-----	|	-----	|	-----	|
|	Creditability	|	Creditability(Target)	|	향후 신용도	|	0 : Bad, 1 : Good	|
|	AccountBalance	|	Account Balance	|	은행잔고	|	1: No account, 2 : None (No balance), 3 : Some Balance	|
|	CreditDuration	|	Duration of Credit (month)	|	신청한 대출기간(월)	|	숫자	|
|	Payment	|	Payment Status of Previous Credit	|	과거 대출 납입 상태	|	0 : 연체, 1 : 기타신용, 2 : 완납, 3 : 정상 대출상환 중 |
|	Purpose	|	Purpose	|	신청한 대출목적	|	1 : New Car , 2 : Used Car , 3 : Furniture , 4 : TV , 5 : Appliances , 6 : Repair , 8 : Vacation , 9 :Retraining , 10 : Business , 0 : Other	|
|	CreditAmount	|	Credit Amount($)	|	신청한 대출금액	|		|
|	Employment	|	Length of current employment(Month)	|	현 직업 근무 기간	|	1: Unemployed,  2: <1 Year,  3: [1, 4),  4: [4, 7),  5: Above 7	|
|	SexMarital	|	Sex & Marital Status	|	성별 & 결혼상태	|	1: Male, Divorced, 2: Male, Single , 3: Male, Married/Widowed , 4: Female	|
|	CurrentAddress	|	Duration in Current address	|	현 거주지 거주기간	|	1: <1 Year , 2: [1, 4) , 3: [4, 7) , 4: Above 7	|
|	MostValuable	|	Most valuable available asset	|	가장 가치있는 자산	|	1: None , 2: Car , 3: Life Insurance , 4: Real Estate	|
|	Age	|	Age (years)	|	나이	|		|
|	AppartmentType	|	Type of apartment	|	주거환경	|	1: free apartment, 2: Rented, 3: Owned	|
|	NoCredits	|	No of Credits at this Bank	|	현재 총 대출 건수	|	1 : one, 2 : 2 ~ 3, 3 : 4 ~ 5, 4 : 6 ~	|
|	Occupation	|	Occupation	|	직업	|	1: Unemployed, unskilled, 2: Unskilled Permanent Resident, 3: Skilled, 4: Executive	|
|	Telephone	|	Telephone	|	전화기 소유 여부	|	2: Yes , 1: No	|
|	ForeignWorker	|	Foreign Worker	|	외국인 근로자 여부	|	2: Yes , 1: No	|


## 2.데이터 준비

### (1) 데이터 정리

In [56]:
data.corr()

,Creditability,AccountBalance,Duration,Payment,Purpose,CreditAmount,Employment,SexMarital,CurrentAddress,MostValuableAsset,Age,Apartment,CreditCount,Occupation,Dependents,Telephone,ForeignWorker
Creditability,1.000000,0.339409,-0.214927,0.227360,-0.017979,-0.154740,0.116002,0.088184,-0.002967,-0.142612,0.091272,0.018119,0.045732,-0.032735,0.003015,0.036466,0.082079
AccountBalance,0.339409,1.000000,-0.071254,0.173076,0.047787,-0.041161,0.088960,0.037549,-0.064727,-0.030244,0.052530,0.029771,0.059144,0.030366,-0.027676,0.063609,-0.036607
Duration,-0.214927,-0.071254,1.000000,-0.067755,0.147492,0.624988,0.057381,0.014789,0.034067,0.303971,-0.037550,0.153126,-0.011284,0.210910,-0.023834,0.164718,-0.134680
Payment,0.227360,0.173076,-0.067755,1.000000,-0.086746,-0.063160,0.113239,0.047001,0.038803,-0.061992,0.116649,0.054989,0.337607,0.023611,0.003580,0.051094,0.012237
Purpose,-0.017979,0.047787,0.147492,-0.086746,1.000000,0.068480,0.016013,0.000157,-0.038221,0.010966,-0.000892,0.013495,0.054935,0.008085,-0.032577,0.078371,-0.113244
CreditAmount,-0.154740,-0.041161,0.624988,-0.063160,0.068480,1.000000,-0.008376,-0.016094,0.028917,0.311602,0.032273,0.133024,0.020785,0.285393,0.017144,0.277000,-0.030662
Employment,0.116002,0.088960,0.057381,0.113239,0.016013,-0.008376,1.000000,0.111278,0.245081,0.087187,0.259116,0.115077,0.125791,0.101225,0.097192,0.060518,-0.022845
SexMarital,0.088184,0.037549,0.014789,0.047001,0.000157,-0.016094,0.111278,1.000000,-0.027269,-0.006940,0.005150,0.098934,0.064672,-0.011956,0.122165,0.027275,0.073103
CurrentAddress,-0.002967,-0.064727,0.034067,0.038803,-0.038221,0.028917,0.245081,-0.027269,1.000000,0.147231,0.265626,0.009990,0.089625,0.012655,0.042643,0.095359,-0.039691
MostValuableAsset,-0.142612,-0.030244,0.303971,-0.061992,0.010966,0.311602,0.087187,-0.006940,0.147231,1.000000,0.074551,0.342969,-0.007765,0.276149,0.011872,0.196802,-0.132462


### (2) 데이터분할1 : x, y 나누기

In [3]:
target = 'Creditability'
x = data.drop(target, axis = 1)
y = data.loc[:, target]

### (3) NA 조치

### (4) 가변수화

In [4]:
dummy_vars = ['Employment', 'CurrentAddress', 'CreditCount', 'Dependents', 'Telephone', 'AccountBalance', 'Payment', 'Purpose', 'SexMarital', 'MostValuableAsset', 'Apartment','Occupation','ForeignWorker']
x = pd.get_dummies(x, columns = dummy_vars, drop_first = True)
x.head()

,Duration,CreditAmount,Age,Employment_2,Employment_3,Employment_4,Employment_5,CurrentAddress_2,CurrentAddress_3,CurrentAddress_4,...,SexMarital_4,MostValuableAsset_2,MostValuableAsset_3,MostValuableAsset_4,Apartment_2,Apartment_3,Occupation_2,Occupation_3,Occupation_4,ForeignWorker_2
0,24,1249,28,1,0,0,0,1,0,0,...,1,0,0,0,1,0,0,1,0,0
1,9,276,22,0,1,0,0,0,0,1,...,1,0,0,0,0,0,1,0,0,0
2,18,1049,21,1,0,0,0,0,0,1,...,0,1,0,0,0,0,0,1,0,0
3,24,6419,44,0,0,0,1,0,0,1,...,0,0,0,1,0,1,0,0,1,0
4,12,1424,55,0,0,0,1,0,0,1,...,0,0,0,0,1,0,0,0,1,0


### (5) 데이터분할2 : train : validation 나누기

In [5]:
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size = .3, random_state = 2022)

### (6) Scaling

In [6]:
scaler = MinMaxScaler()
x_train_s = scaler.fit_transform(x_train)
x_val_s = scaler.transform(x_val)

## 3.모델링

* 사용 알고리즘 : LogisticRegression, DecisionTreeClassifier, KNeighborsClassifier, SVC
* 3가지의 알고리즘을 선정하고 성능을 튜닝해 봅시다.

### (1) 로지스틱 회귀

* 함수 생성

In [57]:
# 아래 함수는 로지스틱 회귀를 위한 전진선택법 함수 입니다.
import statsmodels.api as sm

def forward_stepwise_logistic(x_train, y_train):

    # 변수목록, 선택된 변수 목록, 단계별 모델과 AIC 저장소 정의
    features = list(x_train)
    selected = []
    step_df = pd.DataFrame({ 'step':[], 'feature':[],'aic':[]})

    # 
    for s in range(0, len(features)) :
        result =  { 'step':[], 'feature':[],'aic':[]}

        # 변수 목록에서 변수 한개씩 뽑아서 모델에 추가
        for f in features :
            vars = selected + [f]
            x_tr = x_train[vars]
            model = sm.Logit(y_train, x_tr).fit(disp=False)
            result['step'].append(s+1)
            result['feature'].append(vars)
            result['aic'].append(model.aic)
        
        # 모델별 aic 집계
        temp = pd.DataFrame(result).sort_values('aic').reset_index(drop = True)

        # 만약 이전 aic보다 새로운 aic 가 크다면 멈추기
        if step_df['aic'].min() < temp['aic'].min() :
            break
        step_df = pd.concat([step_df, temp], axis = 0).reset_index(drop = True)

        # 선택된 변수 제거
        v = temp.loc[0,'feature'][s]
        features.remove(v)

        selected.append(v)
    
    # 선택된 변수와 step_df 결과 반환
    return selected, step_df

* 전진선택법 수행

In [58]:
vars, result = forward_stepwise_logistic(x_train, y_train)

* 선택된 변수

In [59]:
vars

['AccountBalance_3',
 'Payment_3',
 'Purpose_1',
 'CreditAmount',
 'Payment_2',
 'Duration',
 'Employment_2',
 'AccountBalance_2',
 'ForeignWorker_2',
 'Purpose_3',
 'SexMarital_3',
 'CreditCount_2',
 'CurrentAddress_2',
 'MostValuableAsset_4',
 'Telephone_2',
 'Occupation_4']

* 전체 변수로 모델링

In [60]:
model1=LogisticRegression()
model1.fit(x_train,y_train)
pred1=model1.predict(x_val)
print(classification_report(y_val,pred1))

              precision    recall  f1-score   support

           0       0.60      0.46      0.52        97
           1       0.77      0.85      0.81       203

    accuracy                           0.73       300
   macro avg       0.68      0.66      0.67       300
weighted avg       0.71      0.73      0.72       300



* 전진선택법 변수로 모델링

In [61]:
model1=LogisticRegression()
model1.fit(x_train[vars],y_train)
pred1=model1.predict(x_val[vars])
print(classification_report(y_val,pred1))

              precision    recall  f1-score   support

           0       0.62      0.46      0.53        97
           1       0.77      0.86      0.81       203

    accuracy                           0.73       300
   macro avg       0.69      0.66      0.67       300
weighted avg       0.72      0.73      0.72       300



### (2) 의사결정나무

In [62]:
params={'max_depth':range(1,10),'min_samples_leaf':range(10,101,10)}

In [63]:
model2=DecisionTreeClassifier()

In [64]:
model2_gs=GridSearchCV(model2,params,cv=5)

In [65]:
model2_gs.fit(x_train,y_train)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(),
             param_grid={'max_depth': range(1, 10),
                         'min_samples_leaf': range(10, 101, 10)})

In [66]:
model2_gs.cv_results_

{'mean_fit_time': array([0.00802193, 0.00441041, 0.00392118, 0.0043962 , 0.0061955 ,
        0.0055788 , 0.00579209, 0.00441275, 0.00529542, 0.00423818,
        0.00457602, 0.0050138 , 0.00419445, 0.00698271, 0.0080368 ,
        0.00809197, 0.00721984, 0.00580735, 0.0045815 , 0.00500846,
        0.00518808, 0.00514827, 0.00562806, 0.00689192, 0.00536242,
        0.00536528, 0.0048346 , 0.00576606, 0.00596581, 0.0050015 ,
        0.00577445, 0.00542889, 0.00569129, 0.00517373, 0.00556273,
        0.00522094, 0.00494123, 0.00536718, 0.00500016, 0.00517802,
        0.00632429, 0.00537782, 0.00596256, 0.00533395, 0.00539632,
        0.00501437, 0.00720863, 0.0053988 , 0.00459671, 0.00445814,
        0.00764761, 0.00607171, 0.00562925, 0.00514121, 0.0049787 ,
        0.00515194, 0.00437241, 0.00438933, 0.00457344, 0.00443497,
        0.00582371, 0.00579405, 0.00646729, 0.00506063, 0.00580263,
        0.00682831, 0.00582161, 0.00461664, 0.00479074, 0.00463214,
        0.00701838, 0.00619202,

In [67]:
model2_gs.best_params_

{'max_depth': 3, 'min_samples_leaf': 80}

In [68]:
model2_gs.best_score_

0.7428571428571429

In [88]:
c

DecisionTreeClassifier(max_depth=3, min_samples_leaf=80)

In [69]:
pred2=model2_gs.predict(x_val)

In [70]:
print(classification_report(y_val,pred2))

              precision    recall  f1-score   support

           0       0.55      0.46      0.50        97
           1       0.76      0.82      0.79       203

    accuracy                           0.70       300
   macro avg       0.66      0.64      0.65       300
weighted avg       0.69      0.70      0.70       300



### (3) KNN

In [71]:
import math

In [72]:
params={'n_neighbors':range(20,40),'metric':['euclidean','manhattan']}

In [73]:
math.sqrt(x_train.shape[0])

26.457513110645905

In [74]:
model3=KNeighborsClassifier()

In [75]:
model3_gs=GridSearchCV(model3,params,cv=5)

In [76]:
model3_gs.fit(x_train_s,y_train)

GridSearchCV(cv=5, estimator=KNeighborsClassifier(),
             param_grid={'metric': ['euclidean', 'manhattan'],
                         'n_neighbors': range(20, 40)})

In [77]:
model3_gs.best_params_

{'metric': 'euclidean', 'n_neighbors': 21}

In [78]:
model3_gs.best_score_

0.7214285714285714

In [79]:
pred3=model3_gs.predict(x_val_s)
print(classification_report(y_val,pred3))

              precision    recall  f1-score   support

           0       0.67      0.08      0.15        97
           1       0.69      0.98      0.81       203

    accuracy                           0.69       300
   macro avg       0.68      0.53      0.48       300
weighted avg       0.68      0.69      0.60       300



### (4) SVM

In [80]:
params={'C':[0.01,0.1,1,10,100],'gamma':np.linspace(0.01,10,20)}

In [81]:
model4=SVC()

In [82]:
model4_gs=GridSearchCV(model4,params,cv=5)

In [83]:
model4_gs.fit(x_train_s,y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.01, 0.1, 1, 10, 100],
                         'gamma': array([ 0.01      ,  0.53578947,  1.06157895,  1.58736842,  2.11315789,
        2.63894737,  3.16473684,  3.69052632,  4.21631579,  4.74210526,
        5.26789474,  5.79368421,  6.31947368,  6.84526316,  7.37105263,
        7.89684211,  8.42263158,  8.94842105,  9.47421053, 10.        ])})

In [84]:
model4_gs.best_params_ # 이근방에서 더 테스트해보기

{'C': 10, 'gamma': 0.01}

In [85]:
model4_gs.best_score_

0.7428571428571429

In [86]:
pred4=model4_gs.predict(x_val_s)

In [87]:
print(classification_report(y_val,pred4))

              precision    recall  f1-score   support

           0       0.71      0.42      0.53        97
           1       0.77      0.92      0.84       203

    accuracy                           0.76       300
   macro avg       0.74      0.67      0.68       300
weighted avg       0.75      0.76      0.74       300

